# One model, three backends

Keras 3 runs the same model on any of the three. This notebook trains it on each in turn and checks that the numbers agree — which is the claim worth verifying rather than assuming.

**Runs on:** CPU — 3 minutes if all three backends are installed &nbsp;·&nbsp; **Slides:** [Chapter 3 — Introduction to TensorFlow, PyTorch, JAX, and Keras](../../../course-web-slides/ch03/index.html) &nbsp;·&nbsp; **Section:** 03 — Keras 3 and the backend switch

---

## Choosing the backend

> ⚠️ **This must happen before `import keras`.** The backend is read at import time; setting it afterwards silently does nothing.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"   # or "torch", or "jax"

import keras
print("keras", keras.__version__, "on", keras.backend.backend())

## keras.ops: NumPy, dispatched

In [ ]:
from keras import ops
import numpy as np

x = ops.array(np.linspace(-3, 3, 7, dtype="float32"))
print("relu   ", ops.relu(x))
print("softmax", ops.softmax(x))
print("mean   ", float(ops.mean(x)))
print("type   ", type(x))

The returned type is the **backend's** tensor type — a `tf.Tensor`, a `torch.Tensor`, or a `jax.Array` — while the API you wrote against is the same. That is the whole trick.

## A model that does not know which backend it is on

In [ ]:
from keras import layers
from keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255
x_test = x_test.reshape(-1, 784).astype("float32") / 255

def build():
    keras.utils.set_random_seed(1337)
    m = keras.Sequential([
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    m.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

model = build()
model.fit(x_train, y_train, epochs=2, batch_size=128, verbose=2)
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"{keras.backend.backend():12s} test accuracy {acc:.4f}")

Expected output:

```
tensorflow   test accuracy 0.96xx
```

## Running the comparison

A backend cannot be changed inside a live process. To compare, run this notebook three times — or drive it from the shell:

In [ ]:
script = '''
import os, sys
os.environ["KERAS_BACKEND"] = sys.argv[1]
import keras
from keras import layers
from keras.datasets import mnist

(x, y), (xt, yt) = mnist.load_data()
x = x.reshape(-1, 784).astype("float32") / 255
xt = xt.reshape(-1, 784).astype("float32") / 255

keras.utils.set_random_seed(1337)
m = keras.Sequential([layers.Dense(128, activation="relu"),
                      layers.Dense(10, activation="softmax")])
m.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
          metrics=["accuracy"])
m.fit(x, y, epochs=2, batch_size=128, verbose=0)
print(sys.argv[1], m.evaluate(xt, yt, verbose=0)[1])
'''
open("_backend_check.py", "w").write(script)
print("now run, in a terminal:")
for b in ["tensorflow", "torch", "jax"]:
    print(f"  python3 _backend_check.py {b}")

> **Note** — Expect the three accuracies to agree to about two decimal places, not exactly. Random number generators differ across backends even under the same seed, and float reductions are not associative — **the same computation in a different order is a different number.**

## Saving on one, loading on another

In [ ]:
model.save("mnist_mlp.keras")

reloaded = keras.saving.load_model("mnist_mlp.keras")
print("reloaded on:", keras.backend.backend())
print("same accuracy:", reloaded.evaluate(x_test, y_test, verbose=0)[1])

The `.keras` format is backend-independent. Train under TensorFlow, serve under PyTorch — this is the practical payoff of the abstraction, and the reason chapter 18 can recommend switching to JAX for one phase of a project without rewriting the model.

---

## What to take away

- `KERAS_BACKEND` must be set **before** `import keras`.
- `keras.ops` is a NumPy-shaped API that dispatches to the backend's own tensors.
- Results agree across backends to about two decimals, not exactly — different RNGs, different reduction orders.
- `.keras` files move between backends, which is what makes the abstraction worth having.